# EventDNA ML Impact Prediction Training Notebook
This notebook walks through the step-by-step training process for the EventDNA GBDT model, utilizing both tabular attributes and Sentence-BERT text descriptions.

## Step 1: Load Data from Database

In [ ]:
import os
import sqlite3
import numpy as np
import pandas as pd

BASE_DIR = os.path.dirname(os.getcwd())
DB_PATH = os.path.join(BASE_DIR, 'event_dna.db')

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query('SELECT * FROM events', conn)
conn.close()
print(f'Loaded {len(df)} events.')

## Step 2: Sentence-BERT Encoding for Unstructured Event Details

In [ ]:
from sentence_transformers import SentenceTransformer

sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
# Build sentence description representation
descriptions = df['event_cause'].astype(str) + ' event at ' + df['junction'].astype(str)
embeddings = sbert_model.encode(descriptions.tolist(), show_progress_bar=True)
print('Embedded description shape:', embeddings.shape)

## Step 3: Tabular Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder
import joblib

categorical_cols = ['event_cause', 'event_type', 'zone', 'junction', 'priority']
encoders = {}
tabular_encoded = []

for col in categorical_cols:
    le = LabelEncoder()
    unique_vals = list(df[col].dropna().unique()) + ['Unknown']
    le.fit(unique_vals)
    df[col] = df[col].apply(lambda x: x if x in le.classes_ else 'Unknown')
    df[col + '_encoded'] = le.transform(df[col])
    encoders[col] = le

## Step 4: Model Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

tabular_features = df[[col + '_encoded' for col in categorical_cols]].values
closure_features = df['requires_road_closure'].values.reshape(-1, 1)
X = np.hstack([tabular_features, closure_features, embeddings])
y = df['impact_score'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

regressor = GradientBoostingRegressor(
    n_estimators=120,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)
regressor.fit(X_train, y_train)

preds = regressor.predict(X_test)
print('RMSE:', np.sqrt(mean_squared_error(y_test, preds)))
print('R² Score:', r2_score(y_test, preds))